In [8]:
# Célula 1 - Importações e reorganizar_dataset (você já tem essa função)
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# reorganizar_dataset deve estar disponível no ambiente conforme seu código fornecido


In [9]:
# Célula 2 - Função para dividir dados e preparar tensores para treinamento e validação

def dividir_dados(caminho_arquivo):
    caminho_arquivo = Path(caminho_arquivo)
    
    if not caminho_arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}")

    df = pd.read_csv(caminho_arquivo, skiprows=1, header=None)
    tempo = df.iloc[:, 0].values.astype(np.float32)  # Coluna de tempo
    dados = df.iloc[:, 1].values.astype(np.float32)
    
    tamanho_treino = int(len(dados) * 0.75)
    treino_tempo = tempo[:tamanho_treino]
    treino_dados = dados[:tamanho_treino]
    validacao_tempo = tempo[tamanho_treino:]
    validacao_dados = dados[tamanho_treino:]
    
    treino_tempo = torch.tensor(treino_tempo).unsqueeze(1)
    treino_dados = torch.tensor(treino_dados).unsqueeze(1)
    validacao_tempo = torch.tensor(validacao_tempo).unsqueeze(1)
    validacao_dados = torch.tensor(validacao_dados).unsqueeze(1)
    
    return {
        "x_train": treino_dados,
        "t_train": treino_tempo,
        "x_val": validacao_dados,
        "t_val": validacao_tempo
    }


In [10]:
# Célula 3 - Classes BaseModel e Autoencoder completas

class BaseModel(nn.Module):
    def __init__(self, **kwargs):
        super(BaseModel, self).__init__()
        self.optimizer = None
        self.criterion = None
        self.kwargs = kwargs

    def compile_model(self, optimizer_fn=optim.Adam, learning_rate=0.001, criterion_fn=nn.MSELoss):
        self.optimizer = optimizer_fn(self.parameters(), lr=learning_rate)
        self.criterion = criterion_fn()

    def train_model(self, x_train, y_train, loss_function, epochs=10, batch_size=32, validation_split=0.2):
        dataset_size = len(x_train)
        split = int(dataset_size * (1 - validation_split))
        train_data = x_train[:split], y_train[:split]
        val_data = x_train[split:], y_train[split:]

        for epoch in range(epochs):
            self.train()
            total_loss = 0
            for i in range(0, split, batch_size):
                x_batch = train_data[0][i:i + batch_size]
                y_batch = train_data[1][i:i + batch_size]

                self.optimizer.zero_grad()
                predictions = self(x_batch)
                loss = loss_function(predictions, y_batch)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            if validation_split > 0:
                self.eval()
                with torch.no_grad():
                    val_predictions = self(val_data[0])
                    val_loss = loss_function(val_predictions, val_data[1])

    def evaluate(self, x_test, y_test, loss_function):
        self.eval()
        with torch.no_grad():
            predictions = self(x_test)
            test_loss = loss_function(predictions, y_test)
        print(f"Test loss: {test_loss.item()}")
        return test_loss.item()

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            predictions = self(x)
        return predictions

    def save_model(self, file_path):
        torch.save(self.state_dict(), file_path)
        print(f"Model saved to {file_path}")

    def load_model(self, file_path):
        self.load_state_dict(torch.load(file_path))
        print(f"Model loaded from {file_path}")


class Autoencoder(BaseModel):
    def __init__(self, **kwargs):
        super(Autoencoder, self).__init__(**kwargs)

        input_dim = kwargs.get("input_dim", 1)
        hidden_dim = kwargs.get("hidden_dim", 32)
        activation_fn = kwargs.get("activation_fn", nn.ReLU)
        dropout = kwargs.get("dropout", 0.2)

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            activation_fn(),
            nn.Dropout(dropout)
        )

        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            activation_fn(),
            nn.Dropout(dropout)
        )

        self.loss_function = nn.MSELoss()

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

    def compile_autoencoder(self, learning_rate=0.001):
        super().compile_model(optimizer_fn=torch.optim.Adam, learning_rate=learning_rate, criterion_fn=nn.MSELoss)
        self.loss_function = nn.MSELoss()

    def train_model(self, x_train, epochs=10, batch_size=32):
        self.train_losses = []
        for epoch in range(epochs):
            self.train()
            total_loss = 0
            for i in range(0, len(x_train), batch_size):
                batch = x_train[i:i+batch_size]
                self.optimizer.zero_grad()
                encoded, decoded = self(batch)
                loss = self.loss_function(decoded, batch)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / (len(x_train) / batch_size)
            self.train_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")

    def reconstruct(self, x):
        self.eval()
        with torch.no_grad():
            _, decoded = self(x)
        return decoded

    def encode(self, x):
        self.eval()
        with torch.no_grad():
            encoded = self.encoder(x)
        return encoded


In [11]:
# Célula 4 - Função para rodar o treinamento, salvar modelos e salvar CSVs conforme pedido

def treinar_e_salvar(model, dataset, epochs=50, batch_size=32, model_path="autoencoder.pth"):
    model.compile_autoencoder(learning_rate=0.001)
    model.train_model(dataset["x_train"], epochs=epochs, batch_size=batch_size)
    model.save_model(model_path)

    # Reconstrução dos dados completos (treino + validação)
    dados_completos = torch.cat([dataset["x_train"], dataset["x_val"]])
    reconstruidos = model.reconstruct(dados_completos).numpy()

    # Espaço latente dos dados completos
    latente = model.encode(dados_completos).numpy()

    # Labels originais concatenados
    labels = np.concatenate([
        np.zeros(len(dataset["x_train"])),  # Supondo labels 0 para treino
        np.zeros(len(dataset["x_val"]))     # Pode adaptar para labels reais
    ])

    # Criar DataFrame para reconstrução (com labels originais)
    df_reconstrucao = pd.DataFrame(reconstruidos, columns=["reconstruido"])
    df_reconstrucao["label"] = labels

    # Criar DataFrame para espaço latente (com labels originais)
    df_latente = pd.DataFrame(latente)
    df_latente["label"] = labels

    # Salvar CSVs
    df_reconstrucao.to_csv("reconstrucao_com_labels.csv", index=False)
    df_latente.to_csv("espaco_latente_com_labels.csv", index=False)

    print("Arquivos CSV salvos:")
    print(" - reconstrução: reconstrução_com_labels.csv")
    print(" - espaço latente: espaco_latente_com_labels.csv")
